In [ ]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))

In [ ]:
import warnings

import torch
import numpy as np
import matplotlib.pyplot as plt

from painter.painter import ActorResNet, RendererFCN
from painter.painter_utils import paint_images
from util.datasets import load_image


warnings.filterwarnings('ignore')

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.device(device)
print(f'device: {device}')

#### Load toy images

In [ ]:
images_dict = {
    'butterfly': load_image(f'../resources/toy_images/n02276258_4835.png'),
    'cat': load_image(f'../resources/toy_images/n02124075_13357.png'),
    'chicken': load_image(f'../resources/toy_images/n01514668_21665.png'),
    'dog': load_image(f'../resources/toy_images/n02110185_1483.png'),
    'elephant': load_image(f'../resources/toy_images/n02504458_650.png'),
    'spider': load_image(f'../resources/toy_images/n01774750_10793.png'),
    'squirrel': load_image(f'../resources/toy_images/n02361337_6774.png')
}

fig, axes = plt.subplots(1, 7, figsize=(20, 5))
for i, d in enumerate(images_dict.items()):
    axes[i].imshow(d[1])
    axes[i].axis('off')
    axes[i].set_title(d[0], fontsize=20)

plt.tight_layout()
plt.show()

## Painting

#### Load pre-trained actor & renderer

In [ ]:
actor_path = f'../resources/models/painter_actor/actor.pkl'
renderer_path = f'../resources/models/painter_renderer/renderer.pkl'
actor = ActorResNet(9, 18, 65) # 65 = 5 (action_bundle) * 13 (stroke parameters)
actor.load_state_dict(torch.load(actor_path))
renderer = RendererFCN()
renderer.load_state_dict(torch.load(renderer_path))

actor = actor.to(device).eval()
renderer = renderer.to(device).eval()
print()

In [ ]:
def prepare_tensor_show(x):
    x = x.detach().to('cpu')
    if x.ndim == 3:
        x = x.permute(1, 2, 0)  # Change (C, H, W) to (H, W, C)
    return x

#### Paint

In [ ]:
output_every = [2, 10, 50, 200, 500, 1000, 5200]
paints_dict = {}
for i, entry_img in enumerate(images_dict.items()):
    print(f'paint image {i+1}...')
    img = torch.tensor(np.expand_dims(entry_img[1], axis=0)).to(device)
    img = img.permute(0, 3, 1, 2)
    canvases = paint_images(x=img,
                            output_every=output_every,
                            device=device,
                            actor=actor,
                            renderer=renderer,
                            add_original=True)
    paints_dict[entry_img[0]] = {}
    for c_i in range(canvases.shape[1]):
        if c_i == canvases.shape[1]-1:
            step = f't=∞'
        else:
            step = f't={output_every[c_i]}'
        paints_dict[entry_img[0]].update({step: canvases[0, c_i]})

#### Output the results

In [ ]:
fig, axes = plt.subplots(7, len(output_every)+1, figsize=(26, 23),
                         gridspec_kw={'wspace': 0, 'hspace': 0, 'left': 0, 'right': 1, 'bottom': 0, 'top': 1})
for i, entry_paints in enumerate(paints_dict.items()):
    animal = entry_paints[0]
    canvases_dict = entry_paints[1]
    items = list(canvases_dict.items())
    items.insert(0, items.pop())
    for j, entry_canvases in enumerate(items):
        t = entry_canvases[0]
        canvas = entry_canvases[1]
        axes[i, j].imshow(prepare_tensor_show(canvas))
        axes[i, j].axis('off')
        if i == 0:
            axes[i, j].set_title(t, fontsize=30)
        j += 1

plt.tight_layout()
plt.show()